# Capítulo 5: Classificação de Cobertura da Terra

**Chris Holden (ceholden@gmail.com) - [https://github.com/ceholden](https://github.com/ceholden)**

-----

## Introdução

Um dos principais usos do Sensoriamento Remoto é mapear a cobertura da terra e as mudanças que ocorrem ao longo do tempo. A maneira mais comum de fazer isso é através de **classificação**, que é o processo de atribuir um rótulo (por exemplo, "floresta", "água") a cada pixel de uma imagem.

No Capítulo 4, criamos uma imagem de Região de Interesse (ROI) que associa um rótulo de classe a alguns dos pixels da nossa imagem raster. Neste capítulo, usaremos esta imagem ROI para treinar um classificador supervisionado e, em seguida, aplicá-lo a todos os pixels da imagem.

Usaremos a biblioteca **Scikit-learn** (também conhecida como `sklearn` ou, anteriormente, `SciKits`), a biblioteca mais popular e bem documentada do Python para aprendizado de máquina.

## Configuração do Colab e Instalação de Bibliotecas

Para este capítulo, precisamos do GDAL/OGR e de todas as principais bibliotecas científicas do Python, incluindo o `scikit-learn` e o `scipy`.

In [ ]:
# Instala as bibliotecas GDAL, NumPy, SciPy e Scikit-learn
!pip install gdal numpy scipy scikit-learn matplotlib --quiet

## Preparação dos Dados

Primeiro, vamos importar o GDAL e o NumPy, carregar nosso *dataset* raster original (`LE70220491999322EDC01_stack.gtif`) e a imagem ROI rasterizada (`training_data.gtif`), e prepará-los para a classificação.

### 1\. Carregar a Imagem e a ROI

In [ ]:
from osgeo import gdal
import numpy as np

# Caminhos simulados para o Colab
RASTER_PATH = '/content/LE70220491999322EDC01_stack.gtif'
ROI_PATH = '/content/training_data.gtif'

# 1. Carregar a Imagem Raster
# NOTE: Em um ambiente real, você abriria o arquivo, mas usaremos uma simulação
# para garantir a execução se os dados de exemplo não estiverem carregados.
try:
    raster_ds = gdal.Open(RASTER_PATH, gdal.GA_ReadOnly)
    roi_ds = gdal.Open(ROI_PATH, gdal.GA_ReadOnly)

    if raster_ds is None or roi_ds is None:
        raise FileNotFoundError("Erro: Um ou ambos os arquivos (Raster/ROI) não foram encontrados. Usando dados simulados.")

    # Carrega todos os dados da imagem (todas as bandas) em um array 3D (Linhas x Colunas x Bandas)
    img = raster_ds.ReadAsArray()

    # Carrega o ROI em um array 2D
    roi = roi_ds.GetRasterBand(1).ReadAsArray()

    # Fecha os datasets do GDAL (liberando a memória)
    raster_ds = None
    roi_ds = None

except FileNotFoundError as e:
    print(e)
    # Geração de dados simulados se os arquivos não forem encontrados
    # (Apenas para fins de demonstração da estrutura do código)
    rows, cols, num_bands = 250, 250, 7
    img = np.random.randint(100, 5000, size=(num_bands, rows, cols), dtype=np.uint16)
    roi = np.zeros((rows, cols), dtype=np.uint8)
    roi[50:100, 50:100] = 1 # Simula a classe 1
    roi[150:200, 150:200] = 2 # Simula a classe 2

    # É importante que a imagem esteja no formato de classificação (B x L x C)
    # O GDAL lê como (Bandas x Linhas x Colunas) por padrão, o que é o esperado pelo código.
    print(f"Usando dados simulados: {img.shape} para Img, {roi.shape} para ROI.")

### 2\. Preparar os Dados de Treinamento

O classificador espera que os dados de treinamento (e os dados a serem classificados) estejam no formato **Feições x Variáveis** (ou Pixels x Bandas). A imagem rasterizada está no formato **Bandas x Linhas x Colunas** (B x L x C), então precisamos:

1.  Transpor a imagem para o formato **L x C x B**.
2.  Achatar a imagem para o formato **(L x C) x B** (Pixels x Bandas).
3.  Achatar a ROI para o formato **(L x C) x 1**.

<!-- end list -->

In [ ]:
# A imagem GDAL é lida no formato (Bandas, Linhas, Colunas).
# Transpõe para (Linhas, Colunas, Bandas) para facilitar o achatamento
img = np.transpose(img, [1, 2, 0])
rows, cols, n_bands = img.shape
print(f"Formato da Imagem Transposta: {img.shape}")

# Achatamos a imagem para o formato (Pixels x Bandas)
X = img.reshape(rows * cols, n_bands)
print(f"Formato Feição-Variável (X): {X.shape}")

# Achatamos o ROI
y = roi.ravel()
print(f"Formato Achatado da ROI (y): {y.shape}")

### 3\. Extrair os Dados para Treinamento

Filtramos os dados `X` e `y` para incluir **apenas** os pixels que foram rotulados na nossa imagem ROI (ou seja, pixels onde `y > 0`).

In [ ]:
# Filtra para incluir apenas os pixels de treinamento (onde a classe não é 0)
is_train = y > 0
X_train = X[is_train, :]
y_train = y[is_train]

print(f"Total de pixels rotulados para treinamento: {len(y_train)}")

# Mapeamento de rótulos para nomes de classes (como definido no Capítulo 4)
labels = {
    1: 'floresta',
    2: 'água',
    3: 'herbáceas',
    4: 'não vegetada', # 'barren'
    5: 'urbana'
}

## Classificação Supervisionada (Random Forest)

O autor opta por um classificador **Random Forest**, um algoritmo poderoso e robusto que funciona bem para a maioria das tarefas de classificação de Sensoriamento Remoto. Usaremos o `RandomForestClassifier` do `scikit-learn`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Cria o modelo (classificador Random Forest)
# n_estimators: número de árvores na floresta (mais é melhor, mas mais lento)
# random_state: para reprodutibilidade
rf = RandomForestClassifier(n_estimators=500, oob_score=True, random_state=42, n_jobs=-1)

# 2. Treina o modelo usando os dados rotulados
print("Iniciando o treinamento do classificador Random Forest...")
rf.fit(X_train, y_train)
print("Treinamento concluído.")

# 3. Avalia o erro Out-of-Bag (OOB)
# O OOB é uma estimativa de erro de validação cruzada para Random Forests
print(f"Erro Out-of-Bag (OOB): {1 - rf.oob_score_:.4f}")

# 4. Exibe a importância das feições (bandas)
print("\nImportância das Bandas na Classificação:")
for i, importance in enumerate(rf.feature_importances_):
    print(f"Banda {i+1}: {importance:.4f}")

## Classificação e Visualização

Com o modelo treinado, aplicaremos ele a **todos** os pixels da imagem (o *array* `X`) e visualizaremos o resultado.

### 1\. Previsão para a Imagem Completa

In [ ]:
print("\nClassificando a imagem completa...")
# O rf.predict(X) retorna um array 1D com os rótulos de classe para cada pixel
Z = rf.predict(X)

# Remodelar o array 1D (Z) de volta para o formato 2D (Linhas x Colunas)
classified_img = Z.reshape(rows, cols)

print("Classificação concluída.")

### 2\. Visualização da Classificação

Usaremos o `matplotlib` para plotar a imagem classificada. Como os rótulos são inteiros (1 a 5), usaremos um mapa de cores discreto.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# Define as classes e cores para o mapa de cores
# O 'class 0' (fundo/NODATA) não está na nossa predição, mas é bom considerá-lo se a ROI tivesse 0.
# Aqui, usamos as 5 classes preditas.
class_values = [1, 2, 3, 4, 5]
# Cores sugeridas:
colors = ['green', 'blue', 'yellow', 'brown', 'red'] # Floresta, Água, Herbácea, Não-Veg, Urbana
cmap = ListedColormap(colors)

# Normalização para mapear valores discretos para o mapa de cores
# As classes de 1 a 5 requerem 6 limites (boundaries): 0.5, 1.5, 2.5, 3.5, 4.5, 5.5
bounds = np.arange(len(class_values) + 1) + 0.5
norm = BoundaryNorm(bounds, cmap.N)

plt.figure(figsize=(10, 10))
img_plot = plt.imshow(classified_img, cmap=cmap, norm=norm)

# Cria a barra de cores com rótulos
cbar = plt.colorbar(img_plot, cmap=cmap, norm=norm, boundaries=bounds, ticks=class_values, orientation='vertical', shrink=0.7)

# Adiciona os rótulos de texto para as classes (1: floresta, etc.)
cbar.set_ticklabels([labels.get(v, f"Classe {v}") for v in class_values])
plt.title('Classificação de Cobertura da Terra (Random Forest)')
plt.xlabel('Colunas')
plt.ylabel('Linhas')
plt.show()

## Conclusão

Neste capítulo, você utilizou o poder das bibliotecas científicas do Python (`scikit-learn` e `NumPy`) em conjunto com os dados geoespaciais (GDAL/OGR) para realizar uma classificação de cobertura da terra. Este é um fluxo de trabalho fundamental em Sensoriamento Remoto, e o Random Forest é um dos métodos mais eficazes e populares para essa tarefa.

-----

Se você gostou deste tutorial, o próximo passo seria calcular a **acurácia** desta classificação (comparando-a com os dados de validação, se disponíveis) e depois aprofundar-se em métodos de processamento espacial (filtros, segmentação), como sugerido no Capítulo 6.

**Parabéns\!** Você completou o ciclo básico de Sensoriamento Remoto e GIS com Python.